In [1]:
import pandas as pd
import numpy as np

In [38]:
PATH_DATA = "data/CIM_ATIH_2025/"
df_icd = pd.read_csv(PATH_DATA +"LIBCIM10MULTI.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd.code = df_icd.code.str.replace(" ","")

In [64]:
PATH_DATA = "data/CepiDc/"

dataset_names = [PATH_DATA +"AlignedCauses_2006-2012full.csv",
                 PATH_DATA +"AlignedCauses_2013full.csv",
                 PATH_DATA +"AlignedCauses_2014_full.csv"]

df =pd.concat(map(lambda file: pd.read_csv(file, sep=";"),
              dataset_names)).rename(columns={"YearCoded":"year",
                                              "Gender":"sex",
                                              "Age":"age",
                                              "ICD10":"code"})



In [65]:
df = df.merge(df_icd[['code','libelle']],how="left")
df.loc[df.libelle.isna(),"libelle"]  = ""

In [66]:
df = df.assign(code = "- " + df.libelle + "(" + df.code +") :" + df.StandardText + "\n")

In [67]:
df.loc[df.code.isna(),"code"]  = ""

In [68]:
df1 = df.sort_values(["DocID","LineID"]).groupby(["DocID"], as_index=True).agg({'code': ' '.join}).reset_index()
df2 = df.sort_values(["DocID","LineID"]).drop_duplicates(subset=["DocID","LineID"]).\
          groupby("DocID", as_index=True).agg({'RawText': ', '.join}).reset_index()

In [71]:
df2.merge(df1,how= "left")

,DocID,RawText,code
0,1,Tableau de mort subite de cause inconnue au te...,
1,2,Carbonisation diffuse avec traumatisme thoraci...,"- Brûlures de parties multiples du corps, au m..."
2,3,Tableau de mort subite au cours d'un effort sp...,- Mort instantanée(R960) :mort subite\n - Surm...
3,4,"arrêt cardio-respiratoire hypoxémique, insuffi...",- Arrêt respiratoire(R092) :arrêt cardio-respi...
4,5,"Oedème pulmonaire aigu, Très probable surdosag...",- Insuffisance ventriculaire gauche(I501) :oap...
...,...,...,...
125378,178384,epidurite metastatique d'un carcinome indiffer...,- Tumeur maligne de siège primitif non précisé...
125379,178385,"métastases hépatiques, cancer de la tête du pa...",- Tumeur maligne secondaire du foie et des voi...
125380,178386,Infarctus du myocarde,"- Infarctus (aigu) du myocarde sans précision,..."
125381,178387,"Anorexie cahexie, Démence , Insuffisance Cardi...",- Cachexie(R64) :cachexie\n - Anorexie(R630) :...
